# TripoSR — environnement GPU temporaire Google Colab

Ce notebook conserve les cellules de vérification du T4 et d'installation des dépendances dans le runtime distant. Il n'installe ni PyTorch, ni CUDA, ni TripoSR sur le PC.

## Limite de service importante

Au 1 novembre 2026, la FAQ officielle de Google Colab classe parmi les activités interdites sur les runtimes gérés l'hébergement/les services web sans lien avec le calcul interactif. Elle précise aussi que, dans l'offre gratuite, l'utilisation d'une interface web sur un runtime Colab pour générer du contenu peut entraîner l'arrêt du runtime. Un serveur Gradio public TripoSR n'est donc pas lancé par ce notebook depuis Colab géré gratuit. Aucun mécanisme de maintien en vie, contournement ou achat n'est proposé.

La FAQ actuelle est consultable à https://research.google.com/colaboratory/faq.html. Le GPU T4 et la disponibilité restent non garantis. La cellule de serveur est volontairement bloquée jusqu'à l'utilisation d'un environnement explicitement autorisé par ses conditions.

Si un serveur Gradio autorisé séparément fournit un lien, la configuration locale attend une ligne `HF_SPACE_ID=<lien>` dans `backend/.env` ; ne la placez jamais dans Git, un document ou un journal. Les liens Gradio partagés sont temporaires (au plus 72 h) et un runtime peut s'arrêter plus tôt.

**Statut : vérification du notebook et dépendances testés sur Colab le 21 octobre 2026 ; redémarrage du serveur non testé le 1 novembre 2026, car Colab a répondu « Échec : backend non attribué ».** Aucun lien actif n'a été reçu lors de cette session. Effacez les sorties avant de versionner le notebook.

Références TripoSR : README, `requirements.txt`, `gradio_app.py` et `run.py` du dépôt officiel VAST-AI-Research/TripoSR. Le code et les poids pré-entraînés TripoSR sont sous licence MIT.


In [ ]:
# Rôle : vérifie le GPU T4 et arrête clairement si le runtime Colab n'en attribue pas.
# Exécution : première cellule de code, avant le clonage et les installations.
import subprocess
import sys
import torch

probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
if probe.returncode != 0 or not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA n'est actif. Colab gratuit ne garantit pas de GPU ; arrêtez ici sans contourner la limite.")
print(probe.stdout)
print(f"Version Python de l'image Colab : {sys.version.split()[0]}")
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU détecté : {gpu_name}")
print(f"Version CUDA fournie avec PyTorch : {torch.version.cuda}")
if "T4" not in gpu_name.upper():
    raise RuntimeError("Ce notebook attend un GPU T4. Arrêtez ici si ce runtime n'est pas un T4.")
print("Vérification T4 réussie. PyTorch vient du runtime Colab ; aucun PyTorch local n'est installé.")


In [ ]:
# Rôle : récupère le dépôt officiel sans écraser un clonage déjà présent.
# Exécution : après vérification du GPU, depuis le runtime Colab temporaire.
from pathlib import Path
import subprocess

repository = Path("/content/TripoSR")
if not repository.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/VAST-AI-Research/TripoSR.git", str(repository)],
        check=True,
    )
elif not (repository / ".git").exists():
    raise RuntimeError(f"{repository} existe mais ne contient pas un dépôt Git ; rien n'a été remplacé.")
else:
    print(f"Dépôt déjà présent : {repository}")
print(f"Code officiel prêt : {repository}")

In [ ]:
# Rôle : installe dans le runtime Colab les dépendances documentées par TripoSR.
# Exécution : après le clonage ; cette installation ne touche pas l'ordinateur local.
import subprocess
import sys

repository = "/content/TripoSR"
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "setuptools"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", f"{repository}/requirements.txt"],
    check=True,
)
print("Installation des dépendances terminée dans le runtime Colab.")

## Dépannage ciblé : erreur CUDA de torchmcubes

N’exécutez la cellule suivante que si l’installation ou l’application échoue explicitement sur `torchmcubes`. Le README de TripoSR donne une commande de réinstallation simple. Le README actuel de `tatsy/torchmcubes` demande PyTorch déjà installé, un compilateur C++20, CMake 3.24 ou plus et, pour le support GPU, CUDA Toolkit 12 ou plus. Il recommande d’installer `scikit-build-core` et `pybind11`, puis de compiler avec `--no-build-isolation` afin de réutiliser le PyTorch du runtime. Sans CUDA Toolkit, le projet annonce une compilation CPU uniquement. Cette cellule affiche stdout/stderr en cas d’échec et ne modifie que Colab.


In [ ]:
# Rôle : réinstalle torchmcubes avec ses prérequis de compilation officiels et conserve le diagnostic pip.
# Exécution : uniquement après un échec explicitement lié à torchmcubes, dans le runtime Colab.
import subprocess
import sys

def installer_et_afficher_erreur(command):
    print("Commande :", " ".join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr pip ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    if result.returncode != 0:
        raise RuntimeError(f"La commande pip a échoué (code {result.returncode}). Consultez stdout et stderr ci-dessus.")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchmcubes"], check=False)
installer_et_afficher_erreur([sys.executable, "-m", "pip", "install", "scikit-build-core", "pybind11", "cmake"])
installer_et_afficher_erreur([
    sys.executable, "-m", "pip", "install", "--no-build-isolation",
    "git+https://github.com/tatsy/torchmcubes.git",
])
print("Réinstallation terminée. Vérifiez les importations avant de démarrer TripoSR.")

In [ ]:
# Rôle : installe les deux compléments nécessaires dans le runtime Colab observé.
# Exécution : après requirements.txt (et le dépannage torchmcubes si nécessaire), avant Gradio.
# Pourquoi : rembg charge onnxruntime ; le hub fourni par requirements.txt est incompatible avec accelerate préinstallé.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "onnxruntime", "huggingface-hub==0.34.0"],
    check=True,
)

import onnxruntime
import torch
import torchmcubes
from huggingface_hub import split_torch_state_dict_into_shards
from transformers.models.vit.modeling_vit import ViTModel

print(f"onnxruntime CPU : {onnxruntime.__version__}")
print(f"torchmcubes : {torchmcubes.__file__}")
print(f"huggingface-hub : {split_torch_state_dict_into_shards.__module__}")
print(f"ViTModel : {ViTModel.__name__}")
print(f"GPU prêt : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'})")


## Démarrer le serveur

La FAQ officielle de Colab interdit sur ses runtimes gérés les services web sans lien avec le calcul interactif et mentionne les interruptions de l'interface web de génération en offre gratuite. Le serveur TripoSR/Gradio public demandé par Forma3D entre dans cette catégorie de risque. Cette étape reste arrêtée sur Colab géré gratuit : ne changez pas le garde-fou et ne cherchez pas à contourner les restrictions.

Un serveur fourni par un environnement explicitement autorisé peut être configuré localement sous la forme `HF_SPACE_ID=<lien>` dans `backend/.env`. Un lien Gradio est public dans cette configuration, temporaire (au plus 72 h), et l'arrêt du processus coupe la génération. Gardez le lien uniquement dans le fichier `.env` local et n'envoyez que des photos autorisées, sans personne ni donnée personnelle.


In [ ]:
# Rôle : empêche le démarrage d'un service web public sur un runtime Colab géré gratuit.
# Exécution : dernière cellule ; le serveur doit être démarré uniquement depuis un environnement autorisé.
raise RuntimeError(
    "Serveur public non lancé : la FAQ officielle de Colab interdit les services web "
    "sans lien avec le calcul interactif sur les runtimes gérés. Utilisez un environnement "
    "explicitement autorisé ; aucun contournement ni maintien en vie n'est fourni."
)
